# 轻量包验收

从包根目录运行。只读取包内科学数据，不读取原 GRN 工作目录、外部 H5AD 或排名数据库。本 notebook 核验最终网络成员、网络与 Top-k 复算、表达五分组、898 个历史 Pearson 记录、六图文件及存储上限。它不重新训练 GRN，也不声称重新验证了外部全量评分的 bin 对齐。

文件完整性、数值一致性与科学外推是不同问题；本验收不将技术重复视为独立生物学样本。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
BASELINE = ROOT / 'data/figure_inputs'
RECOMPUTED = ROOT / 'data/analysis_recomputed'
manifest = json.loads((ROOT / 'manifests/workflow_manifest.json').read_text())
checks = []


def same_table(name, keys):
    reference = pd.read_csv(BASELINE / name, sep='\t').sort_values(keys).reset_index(drop=True)
    result = pd.read_csv(RECOMPUTED / name, sep='\t').sort_values(keys).reset_index(drop=True)
    pd.testing.assert_frame_equal(result[reference.columns], reference, check_dtype=False,
                                  check_exact=False, rtol=1e-10, atol=1e-12)
    checks.append({'check': name, 'status': 'passed', 'rows': len(result)})


counts = []
for spec in [manifest['full'], *manifest['runs']]:
    regulons = json.loads((ROOT / spec['regulons']).read_text())
    edges = pd.read_csv(ROOT / spec['final_edges'], sep='\t')
    expected = {(name, target) for name, targets in regulons.items() for target in targets}
    actual = set(zip(edges.regulon, edges.target))
    assert expected == actual and len(edges) == len(actual)
    assert np.isfinite(edges.weight).all() and edges.run.eq(spec['run']).all()
    counts.append({'run': spec['run'], 'regulons': len(regulons), 'edges': len(edges)})
assert counts[0]['regulons'] == 260 and counts[0]['edges'] == 68492
assert [row['regulons'] for row in counts[1:]] == [220, 221, 213, 218, 212]
for name, keys in [
    ('network_runs.tsv', ['run']),
    ('network_regulons.tsv', ['run', 'regulon']),
    ('network_topk_top100_subset.tsv', ['run', 'k']),
    ('network_topk_top100_subset_per_regulon.tsv', ['run', 'k', 'regulon']),
    ('tf_expression_quintile_group_runs.tsv', ['grouping', 'run', 'quintile']),
    ('tf_expression_quintile_endpoints.tsv', ['run', 'regulon']),
]:
    same_table(name, keys)

features = pd.read_csv(RECOMPUTED / 'tf_expression_quintile_features.tsv', sep='\t')
endpoints = pd.read_csv(RECOMPUTED / 'tf_expression_quintile_endpoints.tsv', sep='\t')
assert len(features) == 260 and features.TF.nunique() == 260
assert features.groupby('expression_quintile').size().eq(52).all()
assert len(endpoints) == 1300 and not endpoints.duplicated(['run', 'regulon']).any()
assert endpoints.pearson.notna().sum() == 898 and endpoints.pearson.isna().sum() == 402
assert endpoints.loc[endpoints.module_recovered.eq(0), 'target_retention'].eq(0).all()
assert endpoints.pearson.notna().eq(endpoints.module_recovered.eq(1)).all()
assert np.allclose(endpoints.target_retention, endpoints.shared_targets / endpoints.n_targets_full)
checks.append({'check': 'TF quintiles and missing-module definitions', 'status': 'passed', 'rows': 1300})
pearson = pd.read_csv(BASELINE / 'native_shared_regulon_pearson.tsv', sep='\t')
assert len(pearson) == 898 and pearson.n_bins.eq(117867).all()
assert np.isfinite(pearson.pearson).all()
assert pearson.groupby('run').size().tolist() == [183, 181, 181, 178, 175]
checks.append({'check': 'provided native Pearson table structure; scores not recomputed', 'status': 'passed', 'rows': 898})
genes = pd.read_csv(BASELINE / 'full_gene_expression_summary.tsv', sep='\t')
assert len(genes) == 15541 and genes.total_counts.sum() == 270918153
assert np.isclose(genes.mean_cp10k.sum(), 10000) and genes.detection_rate.between(0, 1).all()
display(pd.DataFrame(counts), pd.DataFrame(checks))

,run,regulons,edges
0,full,260,68492
1,rep01,220,33270
2,rep02,221,33212
3,rep03,213,33745
4,rep04,218,33381
5,rep05,212,34063


,check,status,rows
0,network_runs.tsv,passed,5
1,network_regulons.tsv,passed,1486
2,network_topk_top100_subset.tsv,passed,25
3,network_topk_top100_subset_per_regulon.tsv,passed,1570
4,tf_expression_quintile_group_runs.tsv,passed,50
5,tf_expression_quintile_endpoints.tsv,passed,1300
6,TF quintiles and missing-module definitions,passed,1300
7,provided native Pearson table structure; score...,passed,898


## 图片、代码与大小

检查六张最终单图的尺寸、Arial TrueType 嵌入及可渲染性。用户提供的旧色阶拼版仅用于确认 caption 的 a–f 顺序，其来源记录于 caption_layout_reference.json，不作为包内最终结果。训练/降采样及外部矩阵分析保留历史输出，本次执行状态由各 notebook 的 reproducibility metadata 和来源清单区分。

In [2]:
import ast
import nbformat
import pymupdf
from PIL import Image
from IPython.core.inputtransformer2 import TransformerManager

names = ['Relative_regulon_count', 'Relative_edge_count', 'Shared_regulon_Pearson',
         'Topk_Jaccard_top100_subset', 'TF_expression_module_recovery', 'TF_expression_target_recovery']
figure_checks = []
for name in names:
    with Image.open(ROOT / 'figures' / (name + '.png')) as image:
        assert image.size == (2700, 2700)
    with pymupdf.open(ROOT / 'figures' / (name + '.pdf')) as pdf:
        assert len(pdf) == 1
        page = pdf[0]
        assert np.allclose([page.rect.width, page.rect.height], [324, 324], atol=.01)
        fonts = page.get_fonts(full=True)
        assert fonts and all('Arial' in font[3] and font[2] != 'Type3' for font in fonts)
        assert not page.get_images() and len(page.get_text()) > 30
        rendered = page.get_pixmap(matrix=pymupdf.Matrix(1, 1), alpha=False)
        assert np.std(np.frombuffer(rendered.samples, dtype=np.uint8)) > 10
        figure_checks.append({'figure': name, 'PDF_pages': 1, 'Arial_embedded': True, 'rendered': True})
plot = nbformat.read(ROOT / 'T577_SpaGRN_visualization.ipynb', as_version=4)
assert plot.metadata.analysis.state == 'complete'
assert sum('image/png' in output.get('data', {}) for cell in plot.cells for output in cell.get('outputs', [])) == 6
assert not [output for cell in plot.cells for output in cell.get('outputs', []) if output.output_type == 'error']
transformer = TransformerManager()
for path in ROOT.rglob('*.ipynb'):
    book = nbformat.read(path, as_version=4)
    nbformat.validate(book)
    for cell in book.cells:
        if cell.cell_type == 'code':
            ast.parse(transformer.transform_cell(cell.source))
files = [path for path in ROOT.rglob('*') if '.git' not in path.relative_to(ROOT).parts and path.is_file()]
symlinks = [path for path in ROOT.rglob('*') if '.git' not in path.relative_to(ROOT).parts and path.is_symlink()]
assert not symlinks
assert all(path.stat().st_size < 10 * 1024 ** 2 for path in files)
assert not any(path.suffix.lower() in ['.h5ad', '.h5', '.feather'] for path in files)
assert sum(path.stat().st_size for path in files) < 100 * 1024 ** 2
report = {
    'status': 'passed', 'data_scope': 'package scientific inputs only',
    'checks': checks, 'GRN_counts': counts, 'figures': figure_checks,
    'size_at_check_bytes': sum(path.stat().st_size for path in files),
    'single_file_limit_bytes': 10 * 1024 ** 2,
    'training_performed': False, 'external_score_matrices_recomputed': False,
    'external_expression_matrix_recomputed': False,
    'history_note': 'Training/downsampling and two external-input analysis notebooks retain original outputs; they were not rerun.',
    'final_inventory': 'manifests/files.tsv generated after validation',
}
(ROOT / 'manifests/validation.json').write_text(json.dumps(report, indent=2))
display(pd.DataFrame(figure_checks), pd.Series({
    'status': report['status'], 'package_MiB_at_check': report['size_at_check_bytes'] / 1024 ** 2,
    'largest_file_MiB': max(path.stat().st_size for path in files) / 1024 ** 2,
    'no_large_matrices_included': True,
}))

,figure,PDF_pages,Arial_embedded,rendered
0,Relative_regulon_count,1,True,True
1,Relative_edge_count,1,True,True
2,Shared_regulon_Pearson,1,True,True
3,Topk_Jaccard_top100_subset,1,True,True
4,TF_expression_module_recovery,1,True,True
5,TF_expression_target_recovery,1,True,True


status                           passed
package_MiB_at_check          54.775712
largest_file_MiB               7.667723
no_large_matrices_included         True
dtype: object